## 0. Установка зависимостей


In [1]:
!pip install -q pandas numpy scipy requests beautifulsoup4 plotly "dash>=2.11" pytrends


zsh:1: command not found: pip


## 1. Импорты


In [2]:
import re
import time
from abc import ABC, abstractmethod

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from bs4 import BeautifulSoup
from scipy import stats

/Users/arsenijandrianov/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. База


In [3]:
class ModuleResult:
    def __init__(self, name):
        self.name = name
        self.tables = {}
        self.figures = {}
        self.findings = []

    def add_finding(self, text):
        self.findings.append(text.strip())


class ResearchModule(ABC):
    def __init__(self, name):
        self.name = name
        self.result = ModuleResult(name)

    @abstractmethod
    def fetch(self): ...

    @abstractmethod
    def transform(self): ...

    @abstractmethod
    def visualize(self): ...

    def run(self):
        self.fetch()
        self.transform()
        self.visualize()
        return self.result

## 3. Оценка выборок и гипотезы


In [4]:
class SampleSummary:
    def __init__(self, n, mean, median, std, ci_low, ci_high):
        self.n = n
        self.mean = mean
        self.median = median
        self.std = std
        self.ci_low = ci_low
        self.ci_high = ci_high


class SampleEvaluator:
    def __init__(self, alpha=0.05):
        self.alpha = alpha

    def summarize(self, values):
        arr = np.asarray(list(values), dtype=float)
        arr = arr[~np.isnan(arr)]
        n = arr.size
        if n == 0:
            return SampleSummary(0, np.nan, np.nan, np.nan, np.nan, np.nan)
        mean = float(arr.mean())
        median = float(np.median(arr))
        std = float(arr.std(ddof=1)) if n > 1 else 0.0
        if n > 1:
            t_crit = stats.t.ppf(1 - self.alpha / 2, df=n - 1)
            margin = t_crit * std / np.sqrt(n)
        else:
            margin = 0.0
        return SampleSummary(n, mean, median, std, mean - margin, mean + margin)

    def remove_outliers_iqr(self, values):
        arr = np.asarray(list(values), dtype=float)
        if arr.size < 4:
            return arr
        q1, q3 = np.percentile(arr, [25, 75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        return arr[(arr >= lo) & (arr <= hi)]


class HypothesisTester:
    def __init__(self, alpha=0.05):
        self.alpha = alpha

    def two_samples_diff(self, sample_a, sample_b, equal_var=False):
        a = np.asarray(list(sample_a), dtype=float)
        b = np.asarray(list(sample_b), dtype=float)
        a = a[~np.isnan(a)]
        b = b[~np.isnan(b)]
        if a.size < 2 or b.size < 2:
            return 0.0, 1.0, False
        t_stat, p_value = stats.ttest_ind(a, b, equal_var=equal_var)
        return float(t_stat), float(p_value), bool(p_value < self.alpha)

## 4. Хелперы для графиков


In [5]:
PALETTE = ["#5B6CFF", "#FF7E7E", "#33C7A4", "#FFB547", "#9B6BFF", "#3B98FF"]


def _layout(fig, title):
    fig.update_layout(
        title=dict(text=title, x=0.02, xanchor="left", font=dict(size=18)),
        template="plotly_white",
        margin=dict(l=40, r=20, t=60, b=40),
        colorway=PALETTE,
        font=dict(family="Inter, Arial, sans-serif", size=13),
    )
    return fig


def build_bar(df, x, y, title, color=None, orientation="v"):
    fig = px.bar(df, x=x, y=y, color=color, orientation=orientation, text_auto=".2s")
    fig.update_traces(textposition="outside", cliponaxis=False)
    return _layout(fig, title)


def build_box(df, x, y, title):
    fig = px.box(df, x=x, y=y, points="suspectedoutliers")
    return _layout(fig, title)


def build_line(df, x, y, title, color=None):
    fig = px.line(df, x=x, y=y, color=color, markers=True)
    return _layout(fig, title)

## 5. Анализ контрагентов

Контрагенты для нас — ландшафт EdTech-игроков: потенциальные платформенные и контентные подрядчики, кандидаты в co-marketing и ориентир по рынку. Парсим их публичные профили, чтобы решить, с кем и почему работать.

In [6]:
WIKI_COMPANIES = [
    "Skillbox",
    "GeekBrains",
    "Нетология",
    "Skyeng",
    "Stepik",
    "Учи.ру",
    "Яндекс_Практикум",
    "Фоксфорд",
    "Maximum_Education",
    "Skillfactory",
    "Яндекс_Учебник",
]


class WikipediaCompanyParser:
    BASE = "https://ru.wikipedia.org/wiki/"
    HEADERS = {"User-Agent": "GlowUpResearch/1.0 (course project)"}

    LABEL_MAP = {
        "основан": "founded",
        "основание": "founded",
        "дата основания": "founded",
        "год основания": "founded",
        "расположение": "headquarters",
        "штаб-квартира": "headquarters",
        "местоположение": "headquarters",
        "число сотрудников": "employees",
        "сотрудников": "employees",
        "отрасль": "industry",
        "тип": "type",
        "выручка": "revenue",
        "владельцы": "owners",
        "ключевые фигуры": "key_people",
        "материнская компания": "parent",
        "сайт": "website",
    }

    def __init__(self, name):
        self.name = name

    def _norm_text(self, value):
        value = re.sub(r"\[\d+\]", "", value)
        return " ".join(value.split())

    def parse(self):
        url = self.BASE + self.name
        response = requests.get(url, headers=self.HEADERS, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        result = {"name": self.name.replace("_", " "), "wiki_url": url}
        infobox = soup.select_one("table.infobox")
        if not infobox:
            return result

        for row in infobox.select("tr"):
            head = row.select_one("th")
            cell = row.select_one("td")
            if not head or not cell:
                continue
            label_raw = self._norm_text(head.get_text(" ", strip=True)).lower()
            key = None
            for prefix, target in self.LABEL_MAP.items():
                if label_raw.startswith(prefix):
                    key = target
                    break
            if not key:
                continue
            result[key] = self._norm_text(cell.get_text(" ", strip=True))
        return result


def parse_wikipedia_companies(names):
    rows = []
    for slug in names:
        try:
            data = WikipediaCompanyParser(slug).parse()
            rows.append(data)
            time.sleep(0.3)
        except Exception as exc:
            print(f"Не удалось обработать {slug}: {exc}")
    return pd.DataFrame(rows)


def extract_year(value):
    if not isinstance(value, str):
        return None
    match = re.search(r"(19|20)\d{2}", value)
    return int(match.group()) if match else None


def extract_headcount(value):
    if not isinstance(value, str):
        return None
    digits = re.sub(r"[^\d]", "", value.split("(")[0])
    return int(digits) if digits else None


`WikipediaCompanyParser` — статический парсер инфобоксов компаний из Wikipedia.


In [7]:
class ContractorsResearch(ResearchModule):
    def __init__(self, company_names=WIKI_COMPANIES):
        super().__init__(name="contractors")
        self.company_names = company_names
        self.evaluator = SampleEvaluator()
        self.raw = None

    def fetch(self):
        self.raw = parse_wikipedia_companies(self.company_names)

    def transform(self):
        df = self.raw.copy()
        df["founded_year"] = df.get("founded", pd.Series(dtype=object)).apply(extract_year)
        df["employees_count"] = df.get("employees", pd.Series(dtype=object)).apply(extract_headcount)

        headquarters = df.get("headquarters", pd.Series("", index=df.index)).fillna("")
        df["is_moscow"] = headquarters.str.contains("Москва", case=False)
        df["age_years"] = df["founded_year"].apply(lambda y: 2026 - int(y) if pd.notna(y) else None)

        self.result.tables["companies"] = df

    def visualize(self):
        df = self.result.tables["companies"]

        plot_df = df.dropna(subset=["employees_count"]).sort_values("employees_count", ascending=False)
        if not plot_df.empty:
            self.result.figures["employees_by_company"] = build_bar(
                plot_df.head(10), x="name", y="employees_count",
                title="Численность штата контрагентов (Wikipedia)",
            )

        msk_share = float(df["is_moscow"].mean()) * 100
        age_summary = self.evaluator.summarize(df["age_years"].dropna())

        self.result.add_finding(
            f"Из {len(df)} разобранных EdTech-контрагентов {msk_share:.0f}% базируются в Москве, "
            "переговоры по платформам и контент-подрядчикам ведём из Москвы без командировок."
        )
        if age_summary.n > 0:
            self.result.add_finding(
                f"Средний возраст контрагента {age_summary.mean:.1f} лет "
                f"(CI95: {age_summary.ci_low:.1f}-{age_summary.ci_high:.1f}). "
                "Рынок зрелый, берём подрядчиков с опытом от 8 лет, так меньше риска."
            )
        if "parent" in df.columns:
            owned = df.dropna(subset=["parent"])
            if not owned.empty:
                self.result.add_finding(
                    f"У {len(owned)} контрагентов есть материнская группа (например, VK, Яндекс), "
                    "с ними надёжнее интеграции и поставки, но условия жёстче."
                )
        big_players = df.dropna(subset=["employees_count"]).query("employees_count >= 500")
        if not big_players.empty:
            self.result.add_finding(
                f"Крупные игроки со штатом 500+: {len(big_players)} "
                f"({', '.join(big_players['name'].head(5).tolist())}). "
                "Их берём ради инфраструктуры и объёма, нишевых ради цены интеграции."
            )

In [8]:
contractors_result = ContractorsResearch().run()
display(contractors_result.tables["companies"].head(10))

for figure in contractors_result.figures.values():
    figure.show()

print("Выводы по разделу:")
for item in contractors_result.findings:
    print(" -", item)


,name,wiki_url,type,founded,headquarters,key_people,industry,parent,website,employees,founded_year,employees_count,is_moscow,age_years
0,Skillbox,https://ru.wikipedia.org/wiki/Skillbox,общество с ограниченной ответственностью,2016,Россия : Москва,Артём Казаков (генеральный директор) [ 2 ],образование ( МСОК : 85 ),Skillbox Holding ( VK ),skillbox.ru,NaN,2016.0,NaN,True,10.0
1,GeekBrains,https://ru.wikipedia.org/wiki/GeekBrains,Общество с ограниченной ответственностью,2010,Россия : Москва,Дмитрий Крутов — генеральный директор [ 1 ],образование ( МСОК : 85 ),Skillbox Holding ( VK ),gb.ru,▲ 568 (2021) [ 2 ],2010.0,568.0,True,16.0
2,Нетология,https://ru.wikipedia.org/wiki/Нетология,частная компания,2011,Россия : Москва,Марианна Снигирева (генеральный директор) Алек...,онлайн-образование,NaN,netology.ru,1500 (2022),2011.0,1500.0,True,15.0
3,Skyeng,https://ru.wikipedia.org/wiki/Skyeng,частная компания,2012,Россия : Москва,Александр Ларьяновский (управляющий партнёр),образование,NaN,skyeng.ru,NaN,2012.0,NaN,True,14.0
4,Stepik,https://ru.wikipedia.org/wiki/Stepik,Онлайн-образование,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
5,Учи.ру,https://ru.wikipedia.org/wiki/Учи.ру,образовательная платформа,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
6,Яндекс Практикум,https://ru.wikipedia.org/wiki/Яндекс_Практикум,Дочернее предприятие,2019,Россия : Москва,Илья Курмышев — генеральный директор,Образование,Яндекс,practicum.yandex.ru,NaN,2019.0,NaN,True,7.0
7,Фоксфорд,https://ru.wikipedia.org/wiki/Фоксфорд,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
8,Maximum Education,https://ru.wikipedia.org/wiki/Maximum_Education,организация,2013,"Москва , Россия",NaN,NaN,NaN,maximumtest.ru (рус.),NaN,2013.0,NaN,True,13.0
9,Skillfactory,https://ru.wikipedia.org/wiki/Skillfactory,Частная компания,2016,Россия : Москва,"Злыгостева Мария (генеральный директор), Суноз...",образование ( МСОК : 85 ),Skillbox Holding ( VK ),skillfactory.ru,NaN,2016.0,NaN,True,10.0


Выводы по разделу:
 - Из 11 разобранных EdTech-контрагентов 64% базируются в Москве, переговоры по платформам и контент-подрядчикам ведём из Москвы без командировок.
 - Средний возраст контрагента 12.1 лет (CI95: 9.1-15.1). Рынок зрелый, берём подрядчиков с опытом от 8 лет, так меньше риска.
 - У 4 контрагентов есть материнская группа (например, VK, Яндекс), с ними надёжнее интеграции и поставки, но условия жёстче.
 - Крупные игроки со штатом 500+: 2 (GeekBrains, Нетология). Их берём ради инфраструктуры и объёма, нишевых ради цены интеграции.


### Закупки и подрядчики

Что именно покупаем для запуска, как часто и почему у этих контрагентов. Сумма SaaS-подписок формирует строку «Платформа», эквайринг — комиссию в P&L (раздел 10).

In [9]:
# Что и у кого закупаем для запуска школы - это наши контрагенты-поставщики.
# Сумма SaaS-подписок даёт строку «Платформа», а эквайринг - комиссию в P&L (раздел 10).
PLATFORM_ITEMS = {
    "LMS-платформа (GetCourse)": 14000,
    "Видеохостинг (Kinescope)": 6000,
    "Email/CRM (Unisender)": 4000,
    "Лендинг + домен (Tilda)": 2000,
    "Аналитика и сервисы": 4000,
}
PLATFORM_COST = sum(PLATFORM_ITEMS.values())   # 30 000 руб./мес - сумма подписок
ACQUIRING_RATE = 0.035                          # ЮKassa, базовый тариф онлайн-эквайринга

procurement = pd.DataFrame([
    {"закупка": "LMS-платформа (GetCourse)", "как часто": "подписка, ежемесячно",
     "стоимость": "14 000 ₽/мес", "почему этот контрагент": "зрелый игрок, готовые интеграции и приём оплат"},
    {"закупка": "Видеохостинг (Kinescope)", "как часто": "подписка, ежемесячно",
     "стоимость": "6 000 ₽/мес", "почему этот контрагент": "российский сервис, видео без блокировок"},
    {"закупка": "Email/CRM (Unisender)", "как часто": "подписка, ежемесячно",
     "стоимость": "4 000 ₽/мес", "почему этот контрагент": "автоворонки прогрева лидов"},
    {"закупка": "Лендинг + домен (Tilda)", "как часто": "подписка, ежемесячно",
     "стоимость": "2 000 ₽/мес", "почему этот контрагент": "быстрый запуск без разработчика"},
    {"закупка": "Аналитика и сервисы", "как часто": "ежемесячно",
     "стоимость": "4 000 ₽/мес", "почему этот контрагент": "трекинг конверсий, чат-боты"},
    {"закупка": "Эквайринг (ЮKassa)", "как часто": "с каждой продажи",
     "стоимость": "3,5% с чека", "почему этот контрагент": "стандартный онлайн-эквайринг"},
    {"закупка": "Компоненты welcome-box", "как часто": "партия на учеников",
     "стоимость": "770 ₽/ученик", "почему этот контрагент": "вовлечение и «вау» (см. раздел 6)"},
    {"закупка": "Эксперты-подрядчики", "как часто": "разово, по гонорару",
     "стоимость": "гонорар", "почему этот контрагент": "гостевые модули косметолога/стилиста"},
])
display(procurement)

platform_df = pd.DataFrame([{"сервис": k, "руб_в_месяц": v} for k, v in PLATFORM_ITEMS.items()])
platform_fig = build_bar(
    platform_df.sort_values("руб_в_месяц", ascending=False),
    x="сервис", y="руб_в_месяц",
    title="Состав расходов на платформу (SaaS-подписки), руб./мес",
)
platform_fig.show()

print(f"Платформа = сумма SaaS-подписок = {PLATFORM_COST:,} руб./мес.")
print(f"Эквайринг = {ACQUIRING_RATE:.1%} с каждой оплаты (ЮKassa).")
print("Это закрывает закупочную часть: что покупаем, как часто, сколько и почему именно у этих контрагентов.")


,закупка,как часто,стоимость,почему этот контрагент
0,LMS-платформа (GetCourse),"подписка, ежемесячно",14 000 ₽/мес,"зрелый игрок, готовые интеграции и приём оплат"
1,Видеохостинг (Kinescope),"подписка, ежемесячно",6 000 ₽/мес,"российский сервис, видео без блокировок"
2,Email/CRM (Unisender),"подписка, ежемесячно",4 000 ₽/мес,автоворонки прогрева лидов
3,Лендинг + домен (Tilda),"подписка, ежемесячно",2 000 ₽/мес,быстрый запуск без разработчика
4,Аналитика и сервисы,ежемесячно,4 000 ₽/мес,"трекинг конверсий, чат-боты"
5,Эквайринг (ЮKassa),с каждой продажи,"3,5% с чека",стандартный онлайн-эквайринг
6,Компоненты welcome-box,партия на учеников,770 ₽/ученик,вовлечение и «вау» (см. раздел 6)
7,Эксперты-подрядчики,"разово, по гонорару",гонорар,гостевые модули косметолога/стилиста


Платформа = сумма SaaS-подписок = 30,000 руб./мес.
Эквайринг = 3.5% с каждой оплаты (ЮKassa).
Это закрывает закупочную часть: что покупаем, как часто, сколько и почему именно у этих контрагентов.


## 6. География компании



In [10]:
CITIES_WIKI_URL = (
    "https://ru.wikipedia.org/wiki/"
    "Список_городов_России_с_населением_более_100_тысяч_человек"
)
HEADERS_WIKI = {"User-Agent": "GlowUpResearch/1.0 (course project)"}

TOP_CITIES = 15
AVERAGE_CHECK = 34_820  # цена курса из анализа конкурентов (Всеволод), 10-й перцентиль рынка

# Состав welcome-box и себестоимость закупки на старте, руб. за единицу.
WELCOME_BOX_ITEMS = {
    "Коробка и упаковка": 70,
    "Печатный гайд-воркбук": 180,
    "Карточки «капсульный гардероб»": 40,
    "Мини-набор по уходу (сэмплы)": 220,
    "Мерч (стикеры, шопер)": 150,
    "Открытка и промокод": 20,
    "Аксессуар (зеркальце/расчёска)": 90,
}


def load_cities_dataset():
    # статический парсинг таблицы населения городов с Wikipedia
    response = requests.get(CITIES_WIKI_URL, headers=HEADERS_WIKI, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.select_one("table.wikitable")
    if table is None:
        raise RuntimeError("Не нашёл таблицу с населением в статье Wikipedia")

    rows = []
    for tr in table.select("tr")[1:]:
        cells = [td.get_text(" ", strip=True) for td in tr.select("td")]
        if len(cells) < 4:
            continue
        city = re.sub(r"\[\d+\]", "", cells[2]).strip()
        last_value = re.sub(r"[^\d]", "", cells[-1])
        if not last_value:
            continue
        rows.append({"city": city, "population_thousand": int(last_value)})

    df = pd.DataFrame(rows).drop_duplicates(subset="city")
    return df.sort_values("population_thousand", ascending=False).head(TOP_CITIES).reset_index(drop=True)


class PochtaTariffFetcher:
    # реальные тарифы Почты России на отправку welcome-box (~1 кг) из Москвы
    ENDPOINT = "https://tariff.pochta.ru/v2/calculate/tariff"
    FROM_INDEX = 105005       # отделение в Москве (пункт приёма)
    OBJECT_PARCEL = 27030     # «Посылка стандарт»
    CITY_INDEX = {
        "Москва": 101000, "Санкт-Петербург": 190000, "Новосибирск": 630000,
        "Екатеринбург": 620000, "Казань": 420000, "Нижний Новгород": 603000,
        "Челябинск": 454000, "Красноярск": 660000, "Самара": 443000, "Уфа": 450000,
        "Ростов-на-Дону": 344000, "Краснодар": 350000, "Омск": 644000,
        "Воронеж": 394000, "Пермь": 614000, "Волгоград": 400000,
    }

    def __init__(self, weight_g=1000):
        self.weight_g = weight_g
        self.cache = {}

    def cost_rub(self, city):
        index = self.CITY_INDEX.get(city)
        if index is None:
            return None
        if index in self.cache:
            return self.cache[index]
        try:
            response = requests.get(self.ENDPOINT, params={
                "json": "", "object": self.OBJECT_PARCEL, "from": self.FROM_INDEX,
                "to": index, "weight": self.weight_g, "pack": 10,
            }, timeout=20)
            pay = response.json().get("paynds")   # стоимость с НДС, в копейках
            if pay is None:
                return None
            value = round(pay / 100.0, 2)
            self.cache[index] = value
            time.sleep(0.2)
            return value
        except Exception as exc:
            print(f"Почта России не ответила для {city}: {exc}")
            return None

In [11]:
class GeographyResearch(ResearchModule):
    STUDIO_CITY = "Москва"

    def __init__(self):
        super().__init__(name="geography")
        self.evaluator = SampleEvaluator()
        self.pochta = PochtaTariffFetcher()

    def fetch(self):
        df = load_cities_dataset()
        # стоимость отправки welcome-box из Москвы в каждый город
        df["delivery_cost_rub"] = df["city"].apply(self.pochta.cost_rub)
        self.cities = df

    def transform(self):
        self.result.tables["cities"] = self.cities.copy()
        box_items = pd.DataFrame(
            [{"компонент": name, "себестоимость_руб": cost}
             for name, cost in WELCOME_BOX_ITEMS.items()]
        ).sort_values("себестоимость_руб", ascending=False).reset_index(drop=True)
        self.result.tables["welcome_box"] = box_items

    def visualize(self):
        df = self.result.tables["cities"]
        with_delivery = df.dropna(subset=["delivery_cost_rub"])

        if not with_delivery.empty:
            self.result.figures["delivery_cost"] = build_bar(
                with_delivery.head(12), x="city", y="delivery_cost_rub",
                title="Стоимость отправки welcome-box из Москвы (Почта России, ~1 кг), руб.",
            )
        self.result.figures["welcome_box_cost"] = build_bar(
            self.result.tables["welcome_box"], x="компонент", y="себестоимость_руб",
            title="Себестоимость наполнения welcome-box, руб.",
        )

        self.result.add_finding(
            "Студию и команду держим в Москве: здесь сосредоточена аудитория (по ЦА 54% - Москва и Петербург) "
            "и отсюда самая дешёвая отправка."
        )
        avg_delivery = 0.0
        if not with_delivery.empty:
            summary = self.evaluator.summarize(with_delivery["delivery_cost_rub"])
            avg_delivery = summary.mean
            self.result.add_finding(
                "Доставка welcome-box Почтой России по стране (посылка ~1 кг из Москвы): "
                f"в среднем {summary.mean:.0f} руб., диапазон "
                f"{with_delivery['delivery_cost_rub'].min():.0f}-{with_delivery['delivery_cost_rub'].max():.0f} руб. "
                "Возим лёгкие разовые посылки, а не фуры, поэтому продаём по всей стране."
            )
        content_cost = float(self.result.tables["welcome_box"]["себестоимость_руб"].sum())
        total_box = content_cost + avg_delivery
        self.result.add_finding(
            f"Себестоимость наполнения welcome-box ~ {content_cost:.0f} руб.; со средней доставкой "
            f"{avg_delivery:.0f} руб. полная стоимость коробки ~ {total_box:.0f} руб. "
            f"(~{total_box / AVERAGE_CHECK * 100:.0f}% от цены курса {AVERAGE_CHECK:.0f} руб.)."
        )

In [12]:
geography_result = GeographyResearch().run()
display(geography_result.tables["cities"].head(10))

for figure in geography_result.figures.values():
    figure.show()

print("Выводы по разделу:")
for item in geography_result.findings:
    print(" -", item)


,city,population_thousand,delivery_cost_rub
0,Москва,13274,330.41
1,Санкт-Петербург,5653,406.66
2,Новосибирск,1637,457.50
3,Екатеринбург,1548,457.50
4,Казань,1330,406.66
5,Красноярск,1212,457.50
6,Нижний Новгород,1198,406.66
7,Челябинск,1177,457.50
8,Уфа,1166,457.50
9,Краснодар,1155,406.66


Выводы по разделу:
 - Студию и команду держим в Москве: здесь сосредоточена аудитория (по ЦА 54% - Москва и Петербург) и отсюда самая дешёвая отправка.
 - Доставка welcome-box Почтой России по стране (посылка ~1 кг из Москвы): в среднем 425 руб., диапазон 330-458 руб. Возим лёгкие разовые посылки, а не фуры, поэтому продаём по всей стране.
 - Себестоимость наполнения welcome-box ~ 770 руб.; со средней доставкой 425 руб. полная стоимость коробки ~ 1195 руб. (~3% от цены курса 34820 руб.).


## 7. Анализ сотрудников



In [13]:
# Зарплаты по ролям берём из открытого API «Работа России» (Роструд): opendata.trudvsem.ru.

ROLE_QUERIES = {
    "Стилист": "стилист",
    "Косметолог": "косметолог",
    "Методист онлайн-курсов": "методист дистанционного обучения",
    "Видеомонтажёр": "видеомонтаж",
    "SMM-менеджер": "smm",
    "Куратор обучения": "куратор обучения",
}

# Страховые взносы с ФОТ в РФ (ПФР+ОМС+ВНиМ), общий тариф по ст. 425 НК РФ.
INSURANCE_RATE = 0.30


class TrudvsemFetcher:
    """Парсер открытого API «Работа России» (Роструд)."""

    ENDPOINT = "http://opendata.trudvsem.ru/api/v1/vacancies"

    def __init__(self, max_pages=3, page_size=100):
        self.max_pages = max_pages
        self.page_size = page_size

    def fetch_role(self, query):
        rows = []
        for page in range(self.max_pages):
            response = requests.get(
                self.ENDPOINT,
                params={"text": query, "limit": self.page_size, "offset": page},
                timeout=25,
            )
            if response.status_code != 200:
                break
            vacancies = (response.json().get("results") or {}).get("vacancies") or []
            if not vacancies:
                break
            for item in vacancies:
                vacancy = item.get("vacancy", {})
                rows.append({
                    "name": vacancy.get("job-name", ""),
                    "city": (vacancy.get("region") or {}).get("name", ""),
                    "company": (vacancy.get("company") or {}).get("name", ""),
                    "salary_from": vacancy.get("salary_min"),
                    "salary_to": vacancy.get("salary_max"),
                    "experience": (vacancy.get("requirement") or {}).get("experience"),
                })
            if len(vacancies) < self.page_size:
                break
            time.sleep(0.2)
        return rows


In [14]:
class EmployeesResearch(ResearchModule):
    def __init__(self):
        super().__init__(name="employees")
        self.evaluator = SampleEvaluator()
        self.tester = HypothesisTester()
        self.fetcher = TrudvsemFetcher()

    def fetch(self):
        rows = []
        for role, query in ROLE_QUERIES.items():
            try:
                for item in self.fetcher.fetch_role(query):
                    rows.append({"role_group": role, **item})
                time.sleep(0.2)
            except Exception as exc:
                print(f"Trudvsem не ответил для {role}: {exc}")
        self.vacancies = pd.DataFrame(rows)
        print(f"Получено вакансий с Trudvsem: {len(self.vacancies)}")

    def transform(self):
        df = self.vacancies.copy()
        if df.empty:
            self.result.tables["vacancies"] = df
            self.result.tables["salary_summary"] = pd.DataFrame()
            return

        df["salary_from"] = pd.to_numeric(df["salary_from"], errors="coerce")
        df["salary_to"] = pd.to_numeric(df["salary_to"], errors="coerce")
        df["salary_avg"] = df[["salary_from", "salary_to"]].mean(axis=1)
        df = df.dropna(subset=["salary_avg"])
        df = df[df["salary_avg"] > 0]

        cleaned_parts = []
        for role, sub in df.groupby("role_group"):
            cleaned = self.evaluator.remove_outliers_iqr(sub["salary_avg"].tolist())
            cleaned_parts.append(sub[sub["salary_avg"].isin(cleaned)])
        df = pd.concat(cleaned_parts, ignore_index=True)

        # полная стоимость сотрудника для компании = оклад + страховые взносы
        df["total_cost"] = df["salary_avg"] * (1 + INSURANCE_RATE)

        salary_summary = (
            df.groupby("role_group")
            .agg(vacancies=("name", "count"),
                 avg_salary=("salary_avg", "mean"),
                 median_salary=("salary_avg", "median"),
                 avg_total_cost=("total_cost", "mean"))
            .reset_index()
            .sort_values("avg_salary", ascending=False)
        )

        self.result.tables["vacancies"] = df
        self.result.tables["salary_summary"] = salary_summary

    def visualize(self):
        salary_summary = self.result.tables.get("salary_summary")
        if salary_summary is None or salary_summary.empty:
            self.result.add_finding("Не удалось получить вакансии по ролям.")
            return
        df = self.result.tables["vacancies"]

        melted = salary_summary.melt(
            id_vars="role_group", value_vars=["avg_salary", "avg_total_cost"],
            var_name="метрика", value_name="руб",
        )
        self.result.figures["salary_vs_cost"] = build_bar(
            melted, x="role_group", y="руб", color="метрика",
            title="Зарплата vs полная стоимость сотрудника (Trudvsem), руб./мес.",
        )
        self.result.figures["salary_box"] = build_box(
            df, x="role_group", y="salary_avg",
            title="Разброс зарплат внутри ролей (Trudvsem)",
        )

        msk = df.loc[df["city"].str.contains("Москва", na=False), "salary_avg"]
        rest = df.loc[~df["city"].str.contains("Москва", na=False), "salary_avg"]
        if msk.size >= 2 and rest.size >= 2:
            t_stat, p_value, reject = self.tester.two_samples_diff(msk, rest)
            self.result.add_finding(
                "Сравнение Москвы и регионов по зарплатам (Trudvsem): "
                "t = {:.2f}, p = {:.3f}. Москва: {:.0f} руб., регионы: {:.0f} руб. {}".format(
                    t_stat, p_value, msk.mean(), rest.mean(),
                    "Разница значима." if reject else "Разница не значима."
                )
            )

        best_role = salary_summary.iloc[0]
        worst_role = salary_summary.iloc[-1]
        self.result.add_finding(
            "Нужные роли и медианная зарплата по рынку: "
            + "; ".join(f"{r['role_group']} - {r['median_salary']:.0f} руб."
                        for _, r in salary_summary.iterrows()) + "."
        )
        self.result.add_finding(
            f"Самая дорогая роль: {best_role['role_group']} "
            f"({best_role['avg_salary']:.0f} руб./мес., полная стоимость с учётом "
            f"30% страховых взносов - {best_role['avg_total_cost']:.0f} руб.). "
            f"Самая доступная: {worst_role['role_group']} ({worst_role['avg_salary']:.0f} руб.)."
        )
        self.result.add_finding(
            f"Всего проанализировано {len(df)} вакансий Trudvsem, выбросы убраны по IQR "
            "внутри каждой роли. Total cost = оклад + 30% (ст. 425 НК РФ); для субъектов МСП "
            "на часть выплат выше МРОТ действует льготная ставка 15%."
        )
        self.result.add_finding(
            "Сверх оклада закладываем добавочные бенефиты как рычаг удержания: ДМС, техника, "
            "оплата обучения. Обязательные страховые взносы (30%) уже включены в полную стоимость."
        )


In [15]:
employees_result = EmployeesResearch().run()
display(employees_result.tables["salary_summary"].round(0))

for figure in employees_result.figures.values():
    figure.show()

print("Выводы по разделу:")
for item in employees_result.findings:
    print(" -", item)


Получено вакансий с Trudvsem: 370


,role_group,vacancies,avg_salary,median_salary,avg_total_cost
2,Косметолог,149,86282.0,75000.0,112167.0
4,Методист онлайн-курсов,7,79286.0,75000.0,103071.0
0,SMM-менеджер,59,67989.0,60000.0,88385.0
5,Стилист,18,54387.0,50000.0,70703.0
3,Куратор обучения,74,51903.0,50000.0,67474.0
1,Видеомонтажёр,30,48433.0,38023.0,62963.0


Выводы по разделу:
 - Сравнение Москвы и регионов по зарплатам (Trudvsem): t = 9.93, p = 0.000. Москва: 104790 руб., регионы: 56975 руб. Разница значима.
 - Нужные роли и медианная зарплата по рынку: Косметолог - 75000 руб.; Методист онлайн-курсов - 75000 руб.; SMM-менеджер - 60000 руб.; Стилист - 50000 руб.; Куратор обучения - 50000 руб.; Видеомонтажёр - 38023 руб..
 - Самая дорогая роль: Косметолог (86282 руб./мес., полная стоимость с учётом 30% страховых взносов - 112167 руб.). Самая доступная: Видеомонтажёр (48433 руб.).
 - Всего проанализировано 337 вакансий Trudvsem, выбросы убраны по IQR внутри каждой роли. Total cost = оклад + 30% (ст. 425 НК РФ); для субъектов МСП на часть выплат выше МРОТ действует льготная ставка 15%.
 - Сверх оклада закладываем добавочные бенефиты как рычаг удержания: ДМС, техника, оплата обучения. Обязательные страховые взносы (30%) уже включены в полную стоимость.


## 8. Маркетинговая кампания


In [16]:
from pytrends.request import TrendReq

KEYWORDS_RU = [
    "уход за лицом",
    "базовый гардероб",
    "looksmaxxing",
    "стилист онлайн",
]


class GoogleTrendsFetcher:
    def __init__(self, geo="RU", timeframe="today 12-m", max_retries=4):
        # повтор запроса при ограничении частоты (HTTP 429)
        self.pytrends = TrendReq(hl="ru-RU", tz=180)
        self.geo = geo
        self.timeframe = timeframe
        self.max_retries = max_retries

    def interest(self, keywords):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                self.pytrends.build_payload(kw_list=keywords, geo=self.geo, timeframe=self.timeframe)
                df = self.pytrends.interest_over_time()
                if df.empty:
                    return df
                if "isPartial" in df.columns:
                    df = df.drop(columns=["isPartial"])
                return df.reset_index()
            except Exception as exc:
                last_error = exc
                time.sleep(2.0 * (attempt + 1))  # пауза перед повтором
        raise last_error

In [17]:
class EdTechMarketParser:
    # оценки объёма рынка берём из статьи Wikipedia «Электронное обучение»
    URL = "https://ru.wikipedia.org/wiki/Электронное_обучение"
    HEADERS = {"User-Agent": "GlowUpResearch/1.0 (course project)"}

    def fetch_text(self):
        response = requests.get(self.URL, headers=self.HEADERS, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        article = soup.select_one("div.mw-parser-output")
        return article.get_text(" ", strip=True) if article else ""

    def extract_market_facts(self, text):
        pattern = re.compile(
            r"([^.\n]*?(\d{1,3}(?:[\s ]\d{3})*|\d+[\.,]?\d*)\s*(млрд|миллиардов|млн|миллионов|трлн)[^.\n]*)",
            re.IGNORECASE,
        )
        keywords = ["рынок", "edtech", "обучен", "образован", "оборот", "индустри", "выручк", "долл"]
        rows = []
        for match in pattern.finditer(text):
            sentence = re.sub(r"\[\s*править[^\]]*\]", "", match.group(1))  # «[ править | править код ]»
            sentence = re.sub(r"\[\s*\d+\s*\]", "", sentence)              # сноски [ 6 ]
            sentence = re.sub(r"\s+", " ", sentence).strip()
            if any(k in sentence.lower() for k in keywords):
                rows.append({"snippet": sentence[:240]})
            if len(rows) >= 8:
                break
        return pd.DataFrame(rows)


In [18]:
class MarketingResearch(ResearchModule):
    KEYWORDS = KEYWORDS_RU

    def __init__(self, average_check=AVERAGE_CHECK, margin=0.55, monthly_budget=300_000, cost_per_lead=300, lead_to_sale=0.07):
        super().__init__(name="marketing")
        # базовые параметры юнит-экономики кампании
        self.average_check = average_check
        self.margin = margin
        self.monthly_budget = monthly_budget
        self.cost_per_lead = cost_per_lead
        self.lead_to_sale = lead_to_sale
        self.trends_fetcher = GoogleTrendsFetcher()
        self.market_parser = EdTechMarketParser()
        self.evaluator = SampleEvaluator()
        self.trends_df = None
        self.market_df = None

    def fetch(self):
        try:
            self.trends_df = self.trends_fetcher.interest(self.KEYWORDS)
        except Exception as exc:
            print("Google Trends недоступен:", exc)
            self.trends_df = pd.DataFrame()

        try:
            text = self.market_parser.fetch_text()
            self.market_df = self.market_parser.extract_market_facts(text)
        except Exception as exc:
            print("Wikipedia EdTech статья недоступна:", exc)
            self.market_df = pd.DataFrame()

    def transform(self):
        if self.trends_df is not None and not self.trends_df.empty:
            tidy = self.trends_df.melt(id_vars="date", var_name="keyword", value_name="interest")

            def summarize_keyword(group):
                group = group.sort_values("date")
                recent = group["interest"].tail(4).mean()        # последние ~4 недели
                prior = group["interest"].iloc[-8:-4].mean()       # предыдущие ~4 недели
                return pd.Series({
                    "avg_interest": group["interest"].mean(),
                    "peak_interest": group["interest"].max(),
                    "recent_mean": recent,
                    "prior_mean": prior,
                    "momentum_pct": (recent - prior) / prior * 100 if prior else 0.0,
                })

            trend_summary = (
                tidy.groupby("keyword")[["date", "interest"]].apply(summarize_keyword)
                .reset_index()
                .sort_values("avg_interest", ascending=False)
            )
            self.result.tables["trends_long"] = tidy
            self.result.tables["trends_summary"] = trend_summary

        if self.market_df is not None and not self.market_df.empty:
            self.result.tables["market_facts"] = self.market_df

    def visualize(self):
        if "trends_long" in self.result.tables:
            tidy = self.result.tables["trends_long"]
            summary = self.result.tables["trends_summary"]

            self.result.figures["trends_dynamics"] = build_line(
                tidy, x="date", y="interest", color="keyword",
                title="Google Trends по нашим темам в России (последние 12 месяцев)",
            )
            self.result.figures["trends_avg"] = build_bar(
                summary, x="keyword", y="avg_interest",
                title="Средний поисковый интерес по темам (Google Trends)",
            )

            top = summary.iloc[0]
            self.result.add_finding(
                f"Самый ёмкий поисковый запрос в России - '{top['keyword']}' "
                f"(средний индекс {top['avg_interest']:.0f}, пик {top['peak_interest']:.0f}). "
                "Делаем под него посадочную страницу и контент-ядро."
            )
            growing = summary[summary["momentum_pct"] > 5]
            cooling = summary[summary["momentum_pct"] < -5]
            if not growing.empty:
                names = ", ".join(f"{r['keyword']} (+{r['momentum_pct']:.0f}%)"
                                  for _, r in growing.iterrows())
                self.result.add_finding(
                    f"Растущие темы (последний месяц vs предыдущий): {names}. На них делаем упор."
                )
            if not cooling.empty:
                names = ", ".join(f"{r['keyword']} ({r['momentum_pct']:.0f}%)"
                                  for _, r in cooling.iterrows())
                self.result.add_finding(
                    f"Темы со снижением интереса: {names}. Используем как поддерживающий контент."
                )

        if "market_facts" in self.result.tables and not self.result.tables["market_facts"].empty:
            preview = self.result.tables["market_facts"]["snippet"].iloc[0]
            self.result.add_finding(
                "Wikipedia (статья 'Электронное обучение') даёт публичные оценки рынка, например: "
                f"'{preview}'. На такие цифры опираемся в инвестпрезентации."
            )

        # прогноз воронки и окупаемости кампании
        max_cac = self.average_check * self.margin                 # предельный CAC = валовая прибыль с продажи
        leads = self.monthly_budget / self.cost_per_lead
        sales = leads * self.lead_to_sale
        actual_cac = self.monthly_budget / sales if sales else float("nan")
        revenue = sales * self.average_check
        romi = (revenue * self.margin - self.monthly_budget) / self.monthly_budget * 100

        self.result.tables["kpi"] = pd.DataFrame([
            {"показатель": "Средний чек, руб", "значение": self.average_check},
            {"показатель": "Маржа", "значение": self.margin},
            {"показатель": "Предельный CAC (= вал. прибыль), руб", "значение": round(max_cac)},
            {"показатель": "Бюджет/мес, руб", "значение": self.monthly_budget},
            {"показатель": "CPL - цена лида, руб", "значение": self.cost_per_lead},
            {"показатель": "Лидов/мес", "значение": round(leads)},
            {"показатель": "Конверсия лид в оплату", "значение": self.lead_to_sale},
            {"показатель": "Продаж/мес", "значение": round(sales)},
            {"показатель": "Фактический CAC, руб", "значение": round(actual_cac)},
            {"показатель": "Выручка/мес, руб", "значение": round(revenue)},
            {"показатель": "ROMI, %", "значение": round(romi)},
        ])
        verdict = ("вписываемся в юнит-экономику" if actual_cac < max_cac
                   else "CAC выше предельного - кампания убыточна")
        self.result.add_finding(
            f"Прогноз кампании при среднем чеке {self.average_check:.0f} руб, марже {self.margin:.0%}, "
            f"бюджете {self.monthly_budget:.0f} руб/мес, цене лида {self.cost_per_lead:.0f} руб и "
            f"конверсии лид в оплату {self.lead_to_sale:.0%}: предельный CAC ~ {max_cac:.0f} руб, "
            f"фактический CAC ~ {actual_cac:.0f} руб, ~{leads:.0f} лидов и ~{sales:.0f} продаж в месяц, "
            f"ROMI ~ {romi:.0f}% ({verdict})."
        )


In [19]:
marketing_result = MarketingResearch().run()
if "trends_summary" in marketing_result.tables:
    display(marketing_result.tables["trends_summary"].round(1))
if "market_facts" in marketing_result.tables:
    display(marketing_result.tables["market_facts"])
if "kpi" in marketing_result.tables:
    display(marketing_result.tables["kpi"])

for figure in marketing_result.figures.values():
    figure.show()

print("Выводы по разделу:")
for item in marketing_result.findings:
    print(" -", item)


/Users/arsenijandrianov/Library/Python/3.9/lib/python/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


,keyword,avg_interest,peak_interest,recent_mean,prior_mean,momentum_pct
3,уход за лицом,66.4,100.0,53.0,74.2,-28.6
1,базовый гардероб,8.4,48.0,12.8,12.0,6.2
0,looksmaxxing,2.1,36.0,15.2,6.2,144.0
2,стилист онлайн,0.4,8.0,5.2,0.0,0.0


,snippet
0,Рынок электронного обучения Мировая индустрия ...
1,Электронное обучение в образовательном сегмент...


,показатель,значение
0,"Средний чек, руб",34820.00
1,Маржа,0.55
2,"Предельный CAC (= вал. прибыль), руб",19151.00
3,"Бюджет/мес, руб",300000.00
4,"CPL - цена лида, руб",300.00
5,Лидов/мес,1000.00
6,Конверсия лид в оплату,0.07
7,Продаж/мес,70.00
8,"Фактический CAC, руб",4286.00
9,"Выручка/мес, руб",2437400.00


Выводы по разделу:
 - Самый ёмкий поисковый запрос в России - 'уход за лицом' (средний индекс 66, пик 100). Делаем под него посадочную страницу и контент-ядро.
 - Растущие темы (последний месяц vs предыдущий): базовый гардероб (+6%), looksmaxxing (+144%). На них делаем упор.
 - Темы со снижением интереса: уход за лицом (-29%). Используем как поддерживающий контент.
 - Wikipedia (статья 'Электронное обучение') даёт публичные оценки рынка, например: 'Рынок электронного обучения Мировая индустрия электронного обучения еще в 2000 году составляла 48 млрд долларов'. На такие цифры опираемся в инвестпрезентации.
 - Прогноз кампании при среднем чеке 34820 руб, марже 55%, бюджете 300000 руб/мес, цене лида 300 руб и конверсии лид в оплату 7%: предельный CAC ~ 19151 руб, фактический CAC ~ 4286 руб, ~1000 лидов и ~70 продаж в месяц, ROMI ~ 347% (вписываемся в юнит-экономику).


## 9. Сводный дашборд на Dash


In [20]:
from dash import Dash, dash_table, dcc, html

PIPELINE_RESULTS = {
    "contractors": contractors_result,
    "geography": geography_result,
    "employees": employees_result,
    "marketing": marketing_result,
}

TABS = [
    ("Контрагенты", "contractors"),
    ("География", "geography"),
    ("Сотрудники", "employees"),
    ("Маркетинг", "marketing"),
]


def _table_card(title, df):
    df_show = df.copy()
    for col in df_show.columns:
        if df_show[col].dtype == "object":
            df_show[col] = df_show[col].astype(str).str.slice(0, 220)
    return html.Div([
        html.H4(title),
        dash_table.DataTable(
            data=df_show.round(2).to_dict("records"),
            columns=[{"name": c, "id": c} for c in df_show.columns],
            style_table={"overflowX": "auto"},
            style_cell={"fontFamily": "Inter, Arial, sans-serif",
                        "fontSize": "13px", "padding": "6px"},
            style_header={"backgroundColor": "#f4f6ff", "fontWeight": "600"},
            page_size=12,
        ),
    ], style={"marginBottom": "24px"})


def _findings_card(findings):
    items = [html.Li(text) for text in findings]
    return html.Div([
        html.H4("Выводы по разделу"),
        html.Ul(items, style={"lineHeight": "1.55"}),
    ], style={"backgroundColor": "#fafbff", "padding": "16px 20px",
              "borderRadius": "12px", "marginBottom": "24px",
              "border": "1px solid #e7ebff"})


def _figures_grid(figures):
    return html.Div(
        [html.Div(dcc.Graph(figure=fig),
                  style={"flex": "1 1 460px", "minWidth": "420px",
                         "marginBottom": "12px"})
         for fig in figures.values()],
        style={"display": "flex", "flexWrap": "wrap", "gap": "12px"},
    )


def build_layout():
    tabs_children = []
    for label, key in TABS:
        module_result = PIPELINE_RESULTS[key]
        tab_content = html.Div([
            _findings_card(module_result.findings),
            _figures_grid(module_result.figures),
            html.Div([_table_card(name, table)
                      for name, table in module_result.tables.items()]),
        ], style={"padding": "12px 4px"})
        tabs_children.append(dcc.Tab(label=label, value=key, children=tab_content))
    return html.Div([
        html.Header([
            html.H1("GlowUp Research: исследование рынка"),
            html.P("Источники данных: Wikipedia, Nominatim, Trudvsem (Роструд), "
                   "Почта России, Google Trends.",
                   style={"color": "#5a607a", "marginTop": 0}),
        ], style={"padding": "24px 28px", "borderBottom": "1px solid #eee"}),
        dcc.Tabs(id="tabs", value=TABS[0][1], children=tabs_children,
                 style={"margin": "0 16px"}),
    ], style={"fontFamily": "Inter, Arial, sans-serif",
              "backgroundColor": "#fff", "minHeight": "100vh"})


app = Dash(__name__, title="GlowUp Research")
app.layout = build_layout()
print("Dash приложение собрано.")


Dash приложение собрано.


In [21]:
import socket
import threading
import time

from IPython.display import IFrame, display


def _pick_free_port():
    """Свободный порт для локального запуска дашборда."""
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.bind(("", 0))
    port = sock.getsockname()[1]
    sock.close()
    return port


DASH_PORT = _pick_free_port()


def _run_app():
    app.run(port=DASH_PORT, debug=False, use_reloader=False)


threading.Thread(target=_run_app, daemon=True).start()
time.sleep(3)

try:
    from google.colab.output import eval_js
    proxy_url = eval_js(f"google.colab.kernel.proxyPort({DASH_PORT})")
    print("Ссылка на дашборд (Colab):", proxy_url)
    display(IFrame(src=proxy_url, width="100%", height=900))
except ImportError:
    local_url = f"http://127.0.0.1:{DASH_PORT}"
    print("Ссылка на дашборд (локально):", local_url)
    display(IFrame(src=local_url, width="100%", height=900))


Ссылка на дашборд (локально): http://127.0.0.1:53242


## 10. Сводная бизнес-модель

Собираем все разделы вместе: штат и ФОТ берём из медиан Trudvsem (раздел 7), стоимость welcome-box — из раздела 6, выручку и бюджет — из KPI маркетинга (раздел 8). Итог — помесячная экономика проекта.

In [22]:
# Стартовый штат под запуск (роль -> число человек).
HEADCOUNT = {
    "Методист онлайн-курсов": 1,
    "Косметолог": 1,
    "Стилист": 1,
    "SMM-менеджер": 1,
    "Видеомонтажёр": 1,
    "Куратор обучения": 2,
}
# PLATFORM_COST и ACQUIRING_RATE заданы в разделе 5 (закупки у контрагентов)

# 1) ФОТ: медианные зарплаты с Trudvsem + 30% страховых взносов
sal = employees_result.tables["salary_summary"].set_index("role_group")
staff_rows = []
for role, people in HEADCOUNT.items():
    median = float(sal.loc[role, "median_salary"]) if role in sal.index else 0.0
    total_one = median * (1 + INSURANCE_RATE)
    staff_rows.append({
        "роль": role,
        "человек": people,
        "оклад_медиана": round(median),
        "полная_стоимость": round(total_one),
        "ФОТ_в_месяц": round(total_one * people),
    })
staff_plan = pd.DataFrame(staff_rows)
monthly_payroll = float(staff_plan["ФОТ_в_месяц"].sum())
headcount_total = int(staff_plan["человек"].sum())

# 2) выручка и продажи из KPI маркетинга
kpi = marketing_result.tables["kpi"].set_index("показатель")["значение"]
sales = float(kpi.get("Продаж/мес", 0))
marketing_budget = float(kpi.get("Бюджет/мес, руб", 0))
margin = float(kpi.get("Маржа", 0.55))
cac = marketing_budget / sales if sales else 0.0
revenue = sales * AVERAGE_CHECK

# 3) себестоимость welcome-box из раздела географии
box_content = float(geography_result.tables["welcome_box"]["себестоимость_руб"].sum())
avg_delivery = float(geography_result.tables["cities"]["delivery_cost_rub"].dropna().mean())
box_cost_one = box_content + avg_delivery
welcome_cogs = box_cost_one * sales

acquiring = revenue * ACQUIRING_RATE
total_costs = monthly_payroll + marketing_budget + welcome_cogs + PLATFORM_COST + acquiring
profit = revenue - total_costs
op_margin = profit / revenue * 100 if revenue else 0.0

# 4) юнит-экономика на одного клиента (переменные затраты на человека)
unit_economics = pd.DataFrame([
    {"строка": "Доход с клиента (чек)", "руб": round(AVERAGE_CHECK)},
    {"строка": "- Привлечение (CAC)", "руб": -round(cac)},
    {"строка": "- Welcome-box", "руб": -round(box_cost_one)},
    {"строка": "- Эквайринг", "руб": -round(AVERAGE_CHECK * ACQUIRING_RATE)},
    {"строка": "= Вклад с клиента", "руб": round(AVERAGE_CHECK - cac - box_cost_one - AVERAGE_CHECK * ACQUIRING_RATE)},
])
contribution_per_customer = AVERAGE_CHECK - cac - box_cost_one - AVERAGE_CHECK * ACQUIRING_RATE
# постоянные расходы (ФОТ + платформа) покрываются суммарным вкладом, а не делятся на клиента
fixed_costs = monthly_payroll + PLATFORM_COST
breakeven_customers = fixed_costs / contribution_per_customer

pnl = pd.DataFrame([
    {"статья": "Выручка", "руб_в_месяц": round(revenue)},
    {"статья": "ФОТ (с взносами)", "руб_в_месяц": -round(monthly_payroll)},
    {"статья": "Маркетинг", "руб_в_месяц": -round(marketing_budget)},
    {"статья": "Welcome-box", "руб_в_месяц": -round(welcome_cogs)},
    {"статья": "Платформа", "руб_в_месяц": -PLATFORM_COST},
    {"статья": "Эквайринг", "руб_в_месяц": -round(acquiring)},
    {"статья": "Прибыль", "руб_в_месяц": round(profit)},
])

# 5) чувствительность к допущениям воронки (бенчмарки таргета RU EdTech)
SCENARIOS = {
    "Пессимистичный": {"cpl": 450, "conv": 0.04},
    "Базовый": {"cpl": 300, "conv": 0.07},
    "Оптимистичный": {"cpl": 200, "conv": 0.10},
}
scenario_rows = []
for name, prm in SCENARIOS.items():
    s_leads = marketing_budget / prm["cpl"]
    s_sales = s_leads * prm["conv"]
    s_rev = s_sales * AVERAGE_CHECK
    s_box = box_cost_one * s_sales
    s_acq = s_rev * ACQUIRING_RATE
    s_profit = s_rev - (monthly_payroll + marketing_budget + s_box + PLATFORM_COST + s_acq)
    s_romi = (s_rev * margin - marketing_budget) / marketing_budget * 100
    scenario_rows.append({
        "сценарий": name, "CPL": prm["cpl"], "конверсия": prm["conv"],
        "продаж_в_месяц": round(s_sales), "CAC": round(marketing_budget / s_sales),
        "ROMI_%": round(s_romi), "прибыль_в_месяц": round(s_profit),
    })
scenarios = pd.DataFrame(scenario_rows)

business = ModuleResult(name="business")
business.tables["staff_plan"] = staff_plan
business.tables["unit_economics"] = unit_economics
business.tables["pnl"] = pnl
business.tables["scenarios"] = scenarios

business.figures["payroll_by_role"] = build_bar(
    staff_plan.sort_values("ФОТ_в_месяц", ascending=False),
    x="роль", y="ФОТ_в_месяц",
    title="Фонд оплаты труда по ролям (оклад + 30% взносов), руб./мес",
)
waterfall = go.Figure(go.Waterfall(
    orientation="v",
    measure=["absolute", "relative", "relative", "relative", "relative", "relative", "total"],
    x=pnl["статья"], y=pnl["руб_в_месяц"],
    text=[f"{v:+,.0f}" for v in pnl["руб_в_месяц"]],
    connector={"line": {"color": "#9AA0CC"}},
    increasing={"marker": {"color": "#33C7A4"}},
    decreasing={"marker": {"color": "#FF7E7E"}},
    totals={"marker": {"color": "#5B6CFF"}},
))
business.figures["pnl_waterfall"] = _layout(waterfall, "Помесячная экономика проекта, руб.")
business.figures["scenarios"] = build_bar(
    scenarios, x="сценарий", y="прибыль_в_месяц",
    title="Операционная прибыль по сценариям воронки, руб./мес",
)

display(staff_plan)
display(unit_economics)
display(pnl)
display(scenarios)
for figure in business.figures.values():
    figure.show()

print(f"Штат под запуск: {headcount_total} человек, ФОТ {monthly_payroll:,.0f} руб./мес.")
print(f"Выручка {revenue:,.0f} руб./мес при {sales:.0f} продажах по среднему чеку {AVERAGE_CHECK:,.0f} руб.")
print(f"Вклад с одного клиента: {contribution_per_customer:,.0f} руб. (чек минус CAC, welcome-box и эквайринг).")
print(f"Операционная прибыль {profit:,.0f} руб./мес ({op_margin:.0f}% маржи), {profit * 12:,.0f} руб./год.")
print(f"Точка безубыточности: {breakeven_customers:.0f} клиентов/мес "
      f"(постоянные {fixed_costs:,.0f} руб. ÷ вклад {contribution_per_customer:,.0f} руб.).")


,роль,человек,оклад_медиана,полная_стоимость,ФОТ_в_месяц
0,Методист онлайн-курсов,1,75000,97500,97500
1,Косметолог,1,75000,97500,97500
2,Стилист,1,50000,65000,65000
3,SMM-менеджер,1,60000,78000,78000
4,Видеомонтажёр,1,38023,49430,49430
5,Куратор обучения,2,50000,65000,130000


,строка,руб
0,Доход с клиента (чек),34820
1,- Привлечение (CAC),-4286
2,- Welcome-box,-1195
3,- Эквайринг,-1219
4,= Вклад с клиента,28120


,статья,руб_в_месяц
0,Выручка,2437400
1,ФОТ (с взносами),-517430
2,Маркетинг,-300000
3,Welcome-box,-83671
4,Платформа,-30000
5,Эквайринг,-85309
6,Прибыль,1420990


,сценарий,CPL,конверсия,продаж_в_месяц,CAC,ROMI_%,прибыль_в_месяц
0,Пессимистичный,450,0.04,27,11250,70,16730
1,Базовый,300,0.07,70,4286,347,1420990
2,Оптимистичный,200,0.10,150,2000,858,4013470


Штат под запуск: 7 человек, ФОТ 517,430 руб./мес.
Выручка 2,437,400 руб./мес при 70 продажах по среднему чеку 34,820 руб.
Вклад с одного клиента: 28,120 руб. (чек минус CAC, welcome-box и эквайринг).
Операционная прибыль 1,420,990 руб./мес (58% маржи), 17,051,878 руб./год.
Точка безубыточности: 19 клиентов/мес (постоянные 547,430 руб. ÷ вклад 28,120 руб.).
